# Post-Adjustment of the MPD Internet-Use Indicator

## Summary

This notebook reuses the final analytical outputs produced by notebook `04` and calibrates them against an external population surface from WorldPop.

The goal is to move from subscription-based MPD internet-use rates toward adult-population estimates while accounting for five structural adjustment parameters defined in `script/conf.py`:

- `CHILDREN_PERC`: the share of the total population excluded from the adult denominator,
- `NO_PHONE_PERC`: the share of adults assumed not to have a phone,
- `SIMS_PER_INTERNET_USER`: the average number of SIMs/subscriptions held by mobile internet users,
- `SIMS_PER_NON_INTERNET_USER`: the average number of SIMs/subscriptions held by mobile non-internet users,
- `NON_MOBILE_PHONE_INTERNET_PERC`: the share of adults without a phone who are still assumed to use the internet.

The logic treats SIM duplication as a bias correction inside the observed mobile subscription sample. It converts the observed internet and non-internet subscription counts into estimated mobile-user counts with separate SIM factors, then recomputes the mobile-user internet share.

The notebook also supports an optional `OPERATOR_CONFIGS` list. If you provide two or more operator-specific notebook `04` output folders together with market shares and operator-specific SIM factors, notebook `05` applies the differential-count correction to each operator table first, then combines the corrected mobile-user percentages using normalized market-share weights and stores operator contribution columns for audit. If `OPERATOR_CONFIGS` is empty, the notebook falls back to a single-operator mode using `INDICATOR_PATH`, `SIMS_PER_INTERNET_USER`, and `SIMS_PER_NON_INTERNET_USER`.

## Core Adjustment Logic

The notebook applies the post-adjustment in six explicit steps:

1. estimate the eligible adult population,
2. estimate the mobile-reachable adult population,
3. correct the observed subscription-level internet share for differential SIM ownership,
4. combine one or more operators through market-share weighting with visible operator contributions,
5. apply phone-ownership coverage to the corrected mobile-user internet share,
6. add the optional non-mobile-phone internet user component and convert the final adjusted share into a final number of internet users.

![Post-adjustment logic](../assets/images/post_adjustment_internet_users_methodology.png)

The municipality logic is therefore written as:

`adult_population = total_population * (1 - CHILDREN_PERC)`

`mobile_reachable_population = adult_population * (1 - NO_PHONE_PERC)`

`corrected_mobile_internet_share_i = (internet_subscriptions_i / sims_per_internet_user_i) / ((internet_subscriptions_i / sims_per_internet_user_i) + (non_internet_subscriptions_i / sims_per_non_internet_user_i))`

`mobile_internet_user_perc = (1 - NO_PHONE_PERC) * sum(market_share_i * corrected_mobile_internet_share_i)`

`non_mobile_internet_user_perc = NO_PHONE_PERC * NON_MOBILE_PHONE_INTERNET_PERC`

`final_internet_user_perc = mobile_internet_user_perc + non_mobile_internet_user_perc`

`final_internet_users = adult_population * final_internet_user_perc`

![Post-adjustment formulas](../assets/images/post_adjustment_internet_users_formulas.png)

With the default `SIMS_PER_INTERNET_USER = 1.1` and `SIMS_PER_NON_INTERNET_USER = 1.0`, the correction makes a modest downward adjustment only because internet users are assumed to hold more SIMs than non-internet users. If both values are equal, the SIM correction cancels out and the observed mobile internet percentage is unchanged.

The notebook stores both the mobile-reachable adult population and the final adjusted internet-use percentage so that the adjustment remains easy to explain and audit.

![Post-adjustment example](../assets/images/post_adjustment_internet_users_example.png)

## Notebook Roadmap

The notebook is organized into six stages:

1. load configuration, validate the post-adjustment parameters, and prepare the output folder,
2. reuse the saved indicator tables from notebook `04`,
3. download or reuse the WorldPop raster and aggregate population to the administrative areas in `GEOJSON_FILE`,
4. calculate adjusted municipality-level internet-user counts and technology splits,
5. reproduce the maps and charts with the adjusted indicators,
6. generate optional zone, age, and gender outputs using the same post-adjustment framework.


## 1. Configure the Environment

This block loads the shared project configuration, creates a dedicated subfolder for the post-adjustment outputs, and defines the working paths used by the notebook.

Unlike notebook `04`, this workflow reads the saved indicator tables directly, so a Spark session is not required. The notebook therefore stays lightweight while still reusing the final results from the previous stage.


In [ ]:
import os
import sys
from pathlib import Path

sys.path.append("../")
sys.path.append("../../")

from script.conf import *

os.makedirs(POST_ADJUSTMENT_PATH, exist_ok=True)
print(f"Post-adjustment indicator outputs will be written to: {POST_ADJUSTMENT_PATH}")

In [ ]:
!{sys.executable} -m pip install rasterio


## 2. Import the Analysis Libraries

The notebook combines tabular processing, spatial joins, charting, and interactive maps. The same visual style used in notebook `04` is applied here so the adjusted outputs remain directly comparable to the raw subscriber-based outputs.


In [ ]:
import base64
import io
import json
import urllib.request
from textwrap import fill

import folium
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import seaborn as sns

from branca.colormap import LinearColormap
from matplotlib.ticker import MultipleLocator
from rasterio.transform import xy
from rasterio.windows import Window, from_bounds
from rasterio.warp import transform as transform_coords, transform_bounds
from shapely.geometry import shape
try:
    from script.geo_utils import load_admin_boundaries as load_clean_admin_boundaries, prepare_web_map_geodata
except ImportError:
    import gzip
    from shapely.geometry import MultiPolygon
    from shapely.ops import transform

    DEFAULT_WEB_SIMPLIFY_TOLERANCE_METERS = 500

    def _force_2d(geometry):
        if geometry is None or geometry.is_empty:
            return geometry
        return transform(lambda x, y, z=None: (x, y), geometry)

    def _polygonal_geometry(geometry):
        if geometry is None or geometry.is_empty:
            return geometry
        if geometry.geom_type == "GeometryCollection":
            polygons = []
            for part in geometry.geoms:
                if part.geom_type == "Polygon":
                    polygons.append(part)
                elif part.geom_type == "MultiPolygon":
                    polygons.extend(part.geoms)
            if not polygons:
                return None
            return polygons[0] if len(polygons) == 1 else MultiPolygon(polygons)
        return geometry

    def load_clean_admin_boundaries(geojson_file, municipality_field_name, municipality_match_name):
        open_func = gzip.open if str(geojson_file).endswith(".gz") else open
        with open_func(geojson_file, "rt", encoding="utf-8") as handle:
            geojson = json.load(handle)

        features = geojson["features"]
        geometries = [
            _polygonal_geometry(_force_2d(shape(feature["geometry"])))
            for feature in features
        ]
        properties = [feature.get("properties", {}) for feature in features]

        geo_df = gpd.GeoDataFrame(properties, geometry=geometries, crs="EPSG:4326")
        geo_df = geo_df.rename(
            columns={
                municipality_field_name: "municipality",
                municipality_match_name: "geomatch",
            }
        )
        geo_df = geo_df[geo_df.geometry.notna() & ~geo_df.geometry.is_empty].copy()

        required_columns = {"municipality", "geomatch", "geometry"}
        missing_columns = required_columns.difference(geo_df.columns)
        if missing_columns:
            raise KeyError(f"Missing expected geometry columns: {sorted(missing_columns)}")

        for column in ["municipality", "geomatch"]:
            geo_df[column] = geo_df[column].astype("string").str.strip()

        return geo_df

    def prepare_web_map_geodata(
        geo_df,
        simplify_tolerance_meters=DEFAULT_WEB_SIMPLIFY_TOLERANCE_METERS,
    ):
        web_geo_df = geo_df.copy()
        web_geo_df["geometry"] = web_geo_df.geometry.map(_force_2d).map(_polygonal_geometry)
        web_geo_df = web_geo_df[web_geo_df.geometry.notna() & ~web_geo_df.geometry.is_empty].copy()

        if simplify_tolerance_meters and simplify_tolerance_meters > 0:
            original_crs = web_geo_df.crs or "EPSG:4326"
            web_geo_df = web_geo_df.to_crs("EPSG:3857")
            web_geo_df["geometry"] = web_geo_df.geometry.simplify(
                simplify_tolerance_meters,
                preserve_topology=True,
            )
            web_geo_df = web_geo_df.to_crs(original_crs)

        return web_geo_df

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.titlesize": 14,
        "axes.titleweight": "bold",
        "axes.labelsize": 11,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
    }
)

## 3. Define Shared Helper Functions

The helper block below performs five jobs:

- validate the percentage and SIM-per-user parameters from `conf.py`,
- load the saved indicator tables from notebook `04`,
- correct subscription-level internet shares for differential SIM ownership,
- optionally combine one or more operator-specific notebook `04` outputs using market-share weights,
- download and process WorldPop using the same URL logic as the synthetic baselayer notebook,
- standardize the adjusted tables, charts, and map layers.

The municipality outputs combine corrected mobile-user internet rates with external population counts. The age and gender outputs do not introduce a separate external demographic population table, so their adjusted counts are allocated from the national adult population using the observed subscriber distribution while their adjusted percentages remain the primary interpretation.

SIM-per-user factors are applied to internet and non-internet subscription counts before recalculating each operator's mobile-user internet share. For two or more operators, those corrected operator-level shares are then combined with normalized market-share weights before the adult-population adjustment.

In [ ]:
TECHNOLOGY_LEVELS = ["2G", "3G", "4G", "5G"]
RAW_PERCENT_COLUMNS = ["perc_no_internet", "perc_internet"] + [
    f"perc_internet_{tech}" for tech in TECHNOLOGY_LEVELS
]
RAW_COUNT_COLUMNS = ["total_home_count", "total_no_internet", "total_internet_ALL"] + [
    f"total_internet_{tech}" for tech in TECHNOLOGY_LEVELS
]
SIM_ADJUSTED_PERCENT_COLUMNS = ["sim_adjusted_perc_no_internet", "sim_adjusted_perc_internet"] + [
    f"sim_adjusted_perc_internet_{tech}" for tech in TECHNOLOGY_LEVELS
]
SIM_ADJUSTED_COUNT_COLUMNS = [
    "sim_adjusted_total_internet_ALL",
    "sim_adjusted_total_no_internet",
    "sim_adjusted_total_home_count",
    "sim_adjusted_total_mobile_users",
] + [f"sim_adjusted_total_internet_{tech}" for tech in TECHNOLOGY_LEVELS]
ADJUSTED_PERCENT_COLUMNS = [
    "adjusted_perc_no_internet",
    "adjusted_perc_internet",
    "adjusted_perc_internet_mobile",
    "adjusted_perc_internet_non_mobile",
] + [f"adjusted_perc_internet_{tech}" for tech in TECHNOLOGY_LEVELS]
MAP_GRADIENT = ["#b2182b", "#ef8a62", "#fddbc7", "#d9f0d3", "#1a9850"]
TECH_COLORS = {
    "No Internet": "#b2182b",
    "Non-mobile": "#4e79a7",
    "2G": "#f28e2b",
    "3G": "#edc948",
    "4G": "#59a14f",
    "5G": "#1a9850",
}
NO_HOME_COLOR = "#d9d9d9"


def validate_percentage_parameter(name, value):
    numeric_value = float(value)
    if not 0 <= numeric_value <= 100:
        raise ValueError(f"{name} must be between 0 and 100. Found: {value}")
    return numeric_value / 100.0


def validate_ratio_parameter(name, value, minimum=0.0):
    numeric_value = float(value)
    if numeric_value < minimum:
        raise ValueError(f"{name} must be greater than or equal to {minimum}. Found: {value}")
    return numeric_value


CHILDREN_SHARE = validate_percentage_parameter("CHILDREN_PERC", CHILDREN_PERC)
NO_PHONE_SHARE = validate_percentage_parameter("NO_PHONE_PERC", NO_PHONE_PERC)
NON_MOBILE_PHONE_INTERNET_SHARE = validate_percentage_parameter(
    "NON_MOBILE_PHONE_INTERNET_PERC",
    NON_MOBILE_PHONE_INTERNET_PERC,
)
CONFIG_SIMS_PER_INTERNET_USER = globals().get("SIMS_PER_INTERNET_USER", 1.0)
CONFIG_SIMS_PER_NON_INTERNET_USER = globals().get("SIMS_PER_NON_INTERNET_USER", 1.0)
DEFAULT_SIMS_PER_INTERNET_USER = validate_ratio_parameter(
    "SIMS_PER_INTERNET_USER",
    CONFIG_SIMS_PER_INTERNET_USER,
    minimum=1.0,
)
DEFAULT_SIMS_PER_NON_INTERNET_USER = validate_ratio_parameter(
    "SIMS_PER_NON_INTERNET_USER",
    CONFIG_SIMS_PER_NON_INTERNET_USER,
    minimum=1.0,
)
SIMS_PER_INTERNET_USER = DEFAULT_SIMS_PER_INTERNET_USER
SIMS_PER_NON_INTERNET_USER = DEFAULT_SIMS_PER_NON_INTERNET_USER


def get_operator_config_value(config, *names, default=None):
    for name in names:
        if name in config and config[name] not in (None, ""):
            return config[name]
    return default


def coerce_operator_configs(configured):
    if isinstance(configured, dict):
        return [configured]
    return list(configured or [])


def resolve_indicator_dir(indicator_dir):
    path = Path(indicator_dir)
    if path.suffix.lower() == ".csv":
        raise ValueError(
            "Operator indicator paths must point to notebook 04 output folders, not individual CSV files. "
            f"Found: {indicator_dir}"
        )
    return path


def build_operator_configs():
    configured = coerce_operator_configs(globals().get("OPERATOR_CONFIGS", []))
    if not configured:
        return [
            {
                "name": "default_operator",
                "indicator_dir": resolve_indicator_dir(INDICATOR_PATH),
                "market_share_weight": 1.0,
                "market_share_perc": 100.0,
                "sims_per_internet_user": DEFAULT_SIMS_PER_INTERNET_USER,
                "sims_per_non_internet_user": DEFAULT_SIMS_PER_NON_INTERNET_USER,
            }
        ]

    market_share_values = [
        get_operator_config_value(config, "market_share_perc", "market_share")
        for config in configured
    ]
    has_market_share = [value is not None for value in market_share_values]
    if any(has_market_share) and not all(has_market_share):
        missing = [
            str(config.get("name", f"operator_{idx}"))
            for idx, (config, has_value) in enumerate(zip(configured, has_market_share), start=1)
            if not has_value
        ]
        raise ValueError(
            "When any OPERATOR_CONFIGS entry provides a market share, every operator must provide one. "
            f"Missing market_share_perc for: {missing}"
        )

    if not any(has_market_share):
        market_share_values = [100.0 / len(configured)] * len(configured)

    operator_records = []
    total_market_share = 0.0
    for idx, (config, market_share_value) in enumerate(zip(configured, market_share_values), start=1):
        name = str(config.get("name", f"operator_{idx}"))
        market_share_perc = float(market_share_value)
        if market_share_perc < 0:
            raise ValueError(f"Market share cannot be negative for {name}. Found: {market_share_perc}")
        sims_per_internet_user = validate_ratio_parameter(
            f"{name} sims_per_internet_user",
            config.get("sims_per_internet_user", config.get("sims_per_user", DEFAULT_SIMS_PER_INTERNET_USER)),
            minimum=1.0,
        )
        sims_per_non_internet_user = validate_ratio_parameter(
            f"{name} sims_per_non_internet_user",
            config.get("sims_per_non_internet_user", DEFAULT_SIMS_PER_NON_INTERNET_USER),
            minimum=1.0,
        )
        indicator_dir = resolve_indicator_dir(get_operator_config_value(
            config,
            "indicator_path",
            "indicator_dir",
            "indicator_folder",
            default=INDICATOR_PATH,
        ))
        total_market_share += market_share_perc
        operator_records.append(
            {
                "name": name,
                "indicator_dir": indicator_dir,
                "market_share_perc": market_share_perc,
                "sims_per_internet_user": sims_per_internet_user,
                "sims_per_non_internet_user": sims_per_non_internet_user,
            }
        )

    if total_market_share <= 0:
        raise ValueError("The OPERATOR_CONFIGS market shares must sum to a value greater than zero.")

    for record in operator_records:
        record["market_share_weight"] = record["market_share_perc"] / total_market_share
    return operator_records


OPERATOR_CONFIG_RECORDS = build_operator_configs()
OPERATOR_MODE = "multi_operator" if len(OPERATOR_CONFIG_RECORDS) > 1 else "single_operator"
NON_MOBILE_INTERNET_COMPONENT_PCT = 100.0 * NO_PHONE_SHARE * NON_MOBILE_PHONE_INTERNET_SHARE


try:
    from script.conf_synthetic import WORLDPOP_YEAR, WORLDPOP_DATABASE_URL, WORLDPOP_USE_1KM, WORLDPOP_UNADJUSTED
except Exception:
    WORLDPOP_YEAR = 2023
    WORLDPOP_DATABASE_URL = "https://data.worldpop.org/"
    WORLDPOP_USE_1KM = False
    WORLDPOP_UNADJUSTED = True


DATA_DIR = Path(BASE_PATH)
INDICATOR_DIR = Path(INDICATOR_PATH)


def load_indicator_table_from_dir(indicator_dir, file_name, required=True):
    table_path = Path(indicator_dir) / file_name
    if not table_path.exists():
        if required:
            raise FileNotFoundError(
                f"Required input table not found: {table_path}. "
                "Run notebook 04 first to generate the saved indicator tables, or update OPERATOR_CONFIGS."
            )
        return None
    print(f"Reading notebook 04 output: {table_path}")
    return pd.read_csv(table_path, sep="\t")


def save_indicator_table(indicator_pdf, file_name):
    output_path = Path(POST_ADJUSTMENT_PATH) / file_name
    output_path.parent.mkdir(parents=True, exist_ok=True)
    indicator_pdf.to_csv(output_path, sep="\t", index=False)
    return output_path


def infer_dimension_columns(indicator_pdf):
    dimension_columns = []
    for column in indicator_pdf.columns:
        if column in RAW_PERCENT_COLUMNS:
            continue
        if column.startswith("perc_") or column.startswith("total_") or column.startswith("adjusted_"):
            continue
        if column.startswith("estimated_") or column.startswith("operator_"):
            continue
        if column == "zone_home_share":
            continue
        dimension_columns.append(column)
    return dimension_columns


def percentage_from_counts(numerator, denominator):
    return np.where(denominator > 0, numerator / denominator * 100.0, 0.0)


def recompute_raw_percentages_from_counts(indicator_pdf, round_output=True):
    recalculated = indicator_pdf.copy()
    if "total_home_count" in recalculated.columns:
        if "total_internet_ALL" in recalculated.columns:
            recalculated["perc_internet"] = percentage_from_counts(
                recalculated["total_internet_ALL"].fillna(0),
                recalculated["total_home_count"].fillna(0),
            )
        if "total_no_internet" in recalculated.columns:
            recalculated["perc_no_internet"] = percentage_from_counts(
                recalculated["total_no_internet"].fillna(0),
                recalculated["total_home_count"].fillna(0),
            )
        for tech in TECHNOLOGY_LEVELS:
            total_col = f"total_internet_{tech}"
            percent_col = f"perc_internet_{tech}"
            if total_col in recalculated.columns:
                recalculated[percent_col] = percentage_from_counts(
                    recalculated[total_col].fillna(0),
                    recalculated["total_home_count"].fillna(0),
                )
    for column in RAW_PERCENT_COLUMNS:
        if column in recalculated.columns:
            recalculated[column] = recalculated[column].fillna(0).clip(0, 100)
            if round_output:
                recalculated[column] = recalculated[column].round(2)
    return recalculated


def apply_sim_duplication_bias_adjustment(
    indicator_pdf,
    sims_per_internet_user,
    sims_per_non_internet_user,
    round_output=True,
):
    adjusted = recompute_raw_percentages_from_counts(indicator_pdf, round_output=round_output)
    required_columns = ["total_internet_ALL", "total_no_internet"]
    missing_columns = [column for column in required_columns if column not in adjusted.columns]
    if missing_columns:
        raise KeyError(
            "SIM duplication bias adjustment requires saved count columns from notebook 04: "
            f"{missing_columns}"
        )

    adjusted["sim_adjusted_total_internet_ALL"] = (
        adjusted["total_internet_ALL"].fillna(0) / sims_per_internet_user
    )
    adjusted["sim_adjusted_total_no_internet"] = (
        adjusted["total_no_internet"].fillna(0) / sims_per_non_internet_user
    )
    adjusted["sim_adjusted_total_home_count"] = (
        adjusted["sim_adjusted_total_internet_ALL"] + adjusted["sim_adjusted_total_no_internet"]
    )
    adjusted["sim_adjusted_total_mobile_users"] = adjusted["sim_adjusted_total_home_count"]

    adjusted["sim_adjusted_perc_internet"] = percentage_from_counts(
        adjusted["sim_adjusted_total_internet_ALL"],
        adjusted["sim_adjusted_total_home_count"],
    )
    adjusted["sim_adjusted_perc_no_internet"] = percentage_from_counts(
        adjusted["sim_adjusted_total_no_internet"],
        adjusted["sim_adjusted_total_home_count"],
    )

    for tech in TECHNOLOGY_LEVELS:
        total_col = f"total_internet_{tech}"
        sim_adjusted_total_col = f"sim_adjusted_total_internet_{tech}"
        sim_adjusted_perc_col = f"sim_adjusted_perc_internet_{tech}"
        if total_col in adjusted.columns:
            adjusted[sim_adjusted_total_col] = adjusted[total_col].fillna(0) / sims_per_internet_user
            adjusted[sim_adjusted_perc_col] = percentage_from_counts(
                adjusted[sim_adjusted_total_col],
                adjusted["sim_adjusted_total_home_count"],
            )

    for column in SIM_ADJUSTED_PERCENT_COLUMNS:
        if column in adjusted.columns:
            adjusted[column] = adjusted[column].fillna(0).clip(0, 100)
            if round_output:
                adjusted[column] = adjusted[column].round(2)

    for column in SIM_ADJUSTED_COUNT_COLUMNS:
        if column in adjusted.columns:
            adjusted[column] = adjusted[column].fillna(0)
            if round_output:
                adjusted[column] = adjusted[column].round(2)

    return adjusted

def apply_operator_sim_adjustment(indicator_pdf, operator, round_output=True):
    return apply_sim_duplication_bias_adjustment(
        indicator_pdf,
        operator["sims_per_internet_user"],
        operator["sims_per_non_internet_user"],
        round_output=round_output,
    )


def combine_operator_indicator_tables(file_name, required=True):
    operator_tables = []
    missing_tables = []
    for operator in OPERATOR_CONFIG_RECORDS:
        indicator_pdf = load_indicator_table_from_dir(operator["indicator_dir"], file_name, required=required)
        if indicator_pdf is None:
            missing_tables.append((operator["name"], Path(operator["indicator_dir"]) / file_name))
            continue
        operator_tables.append((operator, indicator_pdf.copy()))

    if missing_tables:
        missing_labels = ", ".join(f"{name}: {path}" for name, path in missing_tables)
        if operator_tables:
            raise FileNotFoundError(
                f"Some configured operators are missing {file_name}: {missing_labels}. "
                "For multi-operator post-adjustment, each processed output table must be present for every operator."
            )
        if required:
            raise FileNotFoundError(f"Required input table not found for any operator: {missing_labels}")
        return None

    if len(operator_tables) == 1:
        operator, indicator_pdf = operator_tables[0]
        adjusted = apply_operator_sim_adjustment(indicator_pdf, operator)
        adjusted["operator_count"] = 1
        adjusted["operator_market_share_total"] = operator["market_share_perc"]
        adjusted["operator_market_share_weight_total"] = operator["market_share_weight"]
        return adjusted

    key_columns = infer_dimension_columns(operator_tables[0][1])
    for operator, indicator_pdf in operator_tables[1:]:
        other_key_columns = infer_dimension_columns(indicator_pdf)
        if other_key_columns != key_columns:
            raise ValueError(
                f"Dimension columns in {file_name} differ for operator {operator['name']}. "
                f"Expected {key_columns}, found {other_key_columns}."
            )
    merged = None
    for idx, (operator, indicator_pdf) in enumerate(operator_tables):
        operator_pdf = apply_operator_sim_adjustment(indicator_pdf, operator, round_output=False)
        available_columns = [
            column
            for column in RAW_PERCENT_COLUMNS + RAW_COUNT_COLUMNS + SIM_ADJUSTED_PERCENT_COLUMNS + SIM_ADJUSTED_COUNT_COLUMNS
            if column in operator_pdf.columns
        ]
        total_home_count_sum = operator_pdf["total_home_count"].fillna(0).sum() if "total_home_count" in operator_pdf.columns else 0
        if total_home_count_sum > 0:
            operator_pdf["operator_home_share"] = operator_pdf["total_home_count"].fillna(0) / total_home_count_sum
        else:
            operator_pdf["operator_home_share"] = 0.0

        keep_columns = key_columns + available_columns + ["operator_home_share"]
        renamed = operator_pdf[keep_columns].rename(
            columns={
                **{column: f"{column}__op{idx}" for column in available_columns},
                "operator_home_share": f"operator_home_share__op{idx}",
            }
        )
        merged = renamed if merged is None else merged.merge(renamed, on=key_columns, how="outer")

    combined = merged[key_columns].copy()
    combined["operator_count"] = len(operator_tables)
    combined["operator_market_share_total"] = sum(operator["market_share_perc"] for operator, _ in operator_tables)
    combined["operator_market_share_weight_total"] = sum(operator["market_share_weight"] for operator, _ in operator_tables)

    for column in ["perc_internet"] + [f"perc_internet_{tech}" for tech in TECHNOLOGY_LEVELS]:
        combined[column] = 0.0
        for idx, (operator, _) in enumerate(operator_tables):
            source_column = f"{column}__op{idx}"
            contribution_column = f"operator_{idx + 1}_weighted_{column}"
            if source_column in merged.columns:
                contribution = merged[source_column].fillna(0) * operator["market_share_weight"]
                combined[contribution_column] = contribution.round(2)
                combined[column] += contribution
            else:
                combined[contribution_column] = 0.0

    combined["perc_no_internet"] = 100 - combined["perc_internet"]
    for idx, (operator, _) in enumerate(operator_tables):
        source_column = f"perc_no_internet__op{idx}"
        contribution_column = f"operator_{idx + 1}_weighted_perc_no_internet"
        if source_column in merged.columns:
            combined[contribution_column] = (merged[source_column].fillna(0) * operator["market_share_weight"]).round(2)
        else:
            combined[contribution_column] = 0.0

    for column in ["sim_adjusted_perc_internet"] + [
        f"sim_adjusted_perc_internet_{tech}" for tech in TECHNOLOGY_LEVELS
    ]:
        combined[column] = 0.0
        for idx, (operator, _) in enumerate(operator_tables):
            source_column = f"{column}__op{idx}"
            contribution_column = f"operator_{idx + 1}_weighted_{column}"
            if source_column in merged.columns:
                contribution = merged[source_column].fillna(0) * operator["market_share_weight"]
                combined[contribution_column] = contribution.round(2)
                combined[column] += contribution
            else:
                combined[contribution_column] = 0.0

    combined["sim_adjusted_perc_no_internet"] = 100 - combined["sim_adjusted_perc_internet"]
    for idx, (operator, _) in enumerate(operator_tables):
        source_column = f"sim_adjusted_perc_no_internet__op{idx}"
        contribution_column = f"operator_{idx + 1}_weighted_sim_adjusted_perc_no_internet"
        if source_column in merged.columns:
            combined[contribution_column] = (merged[source_column].fillna(0) * operator["market_share_weight"]).round(2)
        else:
            combined[contribution_column] = 0.0

    combined["total_home_count"] = 0.0
    for idx, (operator, _) in enumerate(operator_tables):
        source_column = f"operator_home_share__op{idx}"
        contribution_column = f"operator_{idx + 1}_weighted_home_share"
        if source_column in merged.columns:
            contribution = merged[source_column].fillna(0) * operator["market_share_weight"]
            combined[contribution_column] = contribution.round(6)
            combined["total_home_count"] += contribution
        else:
            combined[contribution_column] = 0.0
    combined["total_home_count"] = (combined["total_home_count"] * 1_000_000).round(0).astype(int)
    combined["total_internet_ALL"] = (combined["total_home_count"] * combined["perc_internet"] / 100.0).round(0).astype(int)
    combined["total_no_internet"] = (combined["total_home_count"] - combined["total_internet_ALL"]).clip(lower=0)
    for tech in TECHNOLOGY_LEVELS:
        percent_col = f"perc_internet_{tech}"
        total_col = f"total_internet_{tech}"
        if percent_col in combined.columns:
            combined[total_col] = (combined["total_home_count"] * combined[percent_col] / 100.0).round(0).astype(int)

    combined["sim_adjusted_total_home_count"] = combined["total_home_count"]
    combined["sim_adjusted_total_mobile_users"] = combined["sim_adjusted_total_home_count"]
    combined["sim_adjusted_total_internet_ALL"] = (
        combined["sim_adjusted_total_home_count"] * combined["sim_adjusted_perc_internet"] / 100.0
    ).round(2)
    combined["sim_adjusted_total_no_internet"] = (
        combined["sim_adjusted_total_home_count"] - combined["sim_adjusted_total_internet_ALL"]
    ).clip(lower=0).round(2)
    for tech in TECHNOLOGY_LEVELS:
        percent_col = f"sim_adjusted_perc_internet_{tech}"
        total_col = f"sim_adjusted_total_internet_{tech}"
        if percent_col in combined.columns:
            combined[total_col] = (
                combined["sim_adjusted_total_home_count"] * combined[percent_col] / 100.0
            ).round(2)

    for column in RAW_PERCENT_COLUMNS + SIM_ADJUSTED_PERCENT_COLUMNS:
        if column in combined.columns:
            combined[column] = combined[column].fillna(0).clip(0, 100).round(2)

    return combined

def load_indicator_table(file_name, required=True):
    return combine_operator_indicator_tables(file_name, required=required)


def ensure_age_group_order(indicator_pdf):
    if indicator_pdf is None or "age_group" not in indicator_pdf.columns:
        return indicator_pdf
    adjusted = indicator_pdf.copy()
    if "age_group_order" not in adjusted.columns:
        adjusted["age_group_order"] = (
            adjusted["age_group"]
            .astype(str)
            .str.extract(r"(\d+)", expand=False)
            .fillna("999")
            .astype(int)
        )
    return adjusted


def build_worldpop_download(pop_year, country_code, worldpop_database_url, pop_res_1km=False, pop_unadj=True):
    if pop_year > 2020:
        folder_name = f"/GIS/Population/Global_2015_2030/R2024B/{pop_year}/{country_code.upper()}/v1/100m/constrained/"
        pop_url = f"{worldpop_database_url}{folder_name}/{country_code.lower()}_pop_{pop_year}_CN_100m_R2024B_v1.tif"
        pop_name = f"{country_code.lower()}_ppp_{pop_year}_100m_UNadj_constrained.tif"
    else:
        folder_name = "GIS/Population/Global_2000_2020/"
        pop_url = (
            f"{worldpop_database_url}{folder_name}{pop_year}/{country_code.upper()}/"
            f"{country_code.lower()}_ppp_{pop_year}"
            f"{('_1km_Aggregated' if pop_res_1km else '')}"
            f"{('_UNadj.tif' if pop_unadj else '.tif')}"
        )
        pop_name = (
            f"{country_code.lower()}_ppp_{pop_year}"
            f"{('_1km_Aggregated' if pop_res_1km else '')}"
            f"{('_UNadj.tif' if pop_unadj else '.tif')}"
        )
    return pop_name, pop_url


def load_admin_boundaries(geojson_file):
    return load_clean_admin_boundaries(
        geojson_file,
        MUNICIPALITY_FIELD_NAME,
        MUNICIPALITY_MATCH_NAME,
    )


def extract_worldpop_points(pop_path, bounds):
    min_lon, min_lat, max_lon, max_lat = bounds

    with rasterio.open(pop_path) as pop_tif:
        print(f"WorldPop raster CRS: {pop_tif.crs}")

        if pop_tif.crs is None:
            raise RuntimeError("WorldPop raster has no CRS; lon/lat bounds cannot be resolved safely.")

        if str(pop_tif.crs).upper() != "EPSG:4326":
            raster_bounds = transform_bounds(
                "EPSG:4326",
                pop_tif.crs,
                min_lon,
                min_lat,
                max_lon,
                max_lat,
                densify_pts=21,
            )
        else:
            raster_bounds = (min_lon, min_lat, max_lon, max_lat)

        raw_window = from_bounds(*raster_bounds, transform=pop_tif.transform)
        row_off = max(0, int(np.floor(raw_window.row_off)))
        col_off = max(0, int(np.floor(raw_window.col_off)))
        row_end = min(pop_tif.height, int(np.ceil(raw_window.row_off + raw_window.height)))
        col_end = min(pop_tif.width, int(np.ceil(raw_window.col_off + raw_window.width)))

        if row_off >= row_end or col_off >= col_end:
            raise ValueError("No WorldPop pixels fall within the administrative bounds.")

        window = Window(
            col_off=col_off,
            row_off=row_off,
            width=col_end - col_off,
            height=row_end - row_off,
        )

        srcband = pop_tif.read(1, window=window, masked=True)
        valid_mask = ~np.ma.getmaskarray(srcband)
        window_transform = pop_tif.window_transform(window)

        rows, cols = np.indices(srcband.shape)
        xs, ys = xy(window_transform, rows, cols, offset="center")
        xs = np.asarray(xs).ravel()
        ys = np.asarray(ys).ravel()

        if str(pop_tif.crs).upper() != "EPSG:4326":
            longitudes, latitudes = transform_coords(
                pop_tif.crs,
                "EPSG:4326",
                xs.tolist(),
                ys.tolist(),
            )
            longitudes = np.asarray(longitudes)
            latitudes = np.asarray(latitudes)
        else:
            longitudes = xs
            latitudes = ys

        population = np.asarray(srcband.filled(0), dtype=float).ravel()

    df_pop = pd.DataFrame(
        {
            "longitude": longitudes,
            "latitude": latitudes,
            "population": population,
        }
    )

    df_pop = df_pop.loc[valid_mask.ravel()].copy()
    df_pop = df_pop[
        df_pop["longitude"].between(min_lon, max_lon)
        & df_pop["latitude"].between(min_lat, max_lat)
    ].copy()
    df_pop = df_pop[df_pop["population"] > 0].reset_index(drop=True)

    if df_pop.empty:
        raise ValueError("No positive WorldPop population pixels remain inside the administrative bounds.")

    return df_pop


def build_constant_series(reference, value):
    return pd.Series(np.full(len(reference), value, dtype=float), index=reference.index)


def get_sim_adjusted_percentage(indicator_pdf, raw_percent_column):
    sim_adjusted_column = raw_percent_column.replace("perc_", "sim_adjusted_perc_", 1)
    if sim_adjusted_column in indicator_pdf.columns:
        return indicator_pdf[sim_adjusted_column]
    return indicator_pdf[raw_percent_column]


def compute_mobile_adjusted_percentage(raw_percentage):
    return raw_percentage.fillna(0) * (1 - NO_PHONE_SHARE)


def compute_non_mobile_adjusted_percentage(reference):
    return build_constant_series(reference, NON_MOBILE_INTERNET_COMPONENT_PCT)


def compute_total_adjusted_percentage(raw_percentage):
    return compute_mobile_adjusted_percentage(raw_percentage) + compute_non_mobile_adjusted_percentage(raw_percentage)


def adjust_area_indicator(indicator_pdf, population_pdf):

    for column in ["municipality", "geomatch"]:
        population_pdf[column] = population_pdf[column].astype("string").str.strip()
        indicator_pdf[column] = indicator_pdf[column].astype("string").str.strip()
    
    adjusted = indicator_pdf.merge(population_pdf, on=["municipality", "geomatch"], how="left")
    adjusted["population_total"] = adjusted["population_total"].fillna(0)
    adjusted["adult_population"] = adjusted["adult_population"].fillna(0)
    adjusted["mobile_reachable_population"] = adjusted["mobile_reachable_population"].fillna(0)

    mobile_internet_percentage = get_sim_adjusted_percentage(adjusted, "perc_internet")
    adjusted["adjusted_perc_internet_mobile"] = compute_mobile_adjusted_percentage(mobile_internet_percentage)
    adjusted["adjusted_perc_internet_non_mobile"] = compute_non_mobile_adjusted_percentage(mobile_internet_percentage)
    adjusted["adjusted_perc_internet"] = adjusted["adjusted_perc_internet_mobile"] + adjusted["adjusted_perc_internet_non_mobile"]
    for tech in TECHNOLOGY_LEVELS:
        raw_col = f"perc_internet_{tech}"
        adjusted_pct_col = f"adjusted_perc_internet_{tech}"
        adjusted[adjusted_pct_col] = compute_mobile_adjusted_percentage(
            get_sim_adjusted_percentage(adjusted, raw_col)
        )
        adjusted[f"adjusted_total_internet_{tech}"] = (
            adjusted["adult_population"] * adjusted[adjusted_pct_col] / 100.0
        )

    adjusted["adjusted_total_internet_non_mobile"] = (
        adjusted["adult_population"] * adjusted["adjusted_perc_internet_non_mobile"] / 100.0
    )
    adjusted["adjusted_total_internet_ALL"] = (
        adjusted["adult_population"] * adjusted["adjusted_perc_internet"] / 100.0
    )
    adjusted["adjusted_total_internet_mobile"] = (
        adjusted["adjusted_total_internet_ALL"] - adjusted["adjusted_total_internet_non_mobile"]
    ).clip(lower=0)
    adjusted["adjusted_total_no_internet"] = (
        adjusted["adult_population"] - adjusted["adjusted_total_internet_ALL"]
    ).clip(lower=0)
    adjusted["adjusted_perc_no_internet"] = np.where(
        adjusted["adult_population"] > 0,
        adjusted["adjusted_total_no_internet"] / adjusted["adult_population"] * 100,
        0,
    )

    count_columns = [
        "population_total",
        "adult_population",
        "mobile_reachable_population",
        "adjusted_total_internet_mobile",
        "adjusted_total_internet_non_mobile",
        "adjusted_total_internet_ALL",
        "adjusted_total_no_internet",
    ] + [f"adjusted_total_internet_{tech}" for tech in TECHNOLOGY_LEVELS]

    for column in count_columns:
        adjusted[column] = adjusted[column].fillna(0).round(0).astype(int)

    for column in ADJUSTED_PERCENT_COLUMNS:
        adjusted[column] = adjusted[column].fillna(0).clip(0, 100).round(2)

    return adjusted.sort_values(["adjusted_total_internet_ALL", "municipality"], ascending=[False, True]).reset_index(drop=True)


def adjust_segment_percentages(indicator_pdf):
    adjusted = indicator_pdf.copy()
    mobile_internet_percentage = get_sim_adjusted_percentage(adjusted, "perc_internet")
    adjusted["adjusted_perc_internet_mobile"] = compute_mobile_adjusted_percentage(mobile_internet_percentage)
    adjusted["adjusted_perc_internet_non_mobile"] = compute_non_mobile_adjusted_percentage(mobile_internet_percentage)
    adjusted["adjusted_perc_internet"] = adjusted["adjusted_perc_internet_mobile"] + adjusted["adjusted_perc_internet_non_mobile"]
    for tech in TECHNOLOGY_LEVELS:
        adjusted[f"adjusted_perc_internet_{tech}"] = compute_mobile_adjusted_percentage(
            get_sim_adjusted_percentage(adjusted, f"perc_internet_{tech}")
        )
    adjusted["adjusted_perc_no_internet"] = 100 - adjusted["adjusted_perc_internet"]

    for column in ADJUSTED_PERCENT_COLUMNS:
        adjusted[column] = adjusted[column].fillna(0).clip(0, 100).round(2)

    return adjusted


def allocate_population_from_subscriber_share(indicator_pdf, adult_population_total, mobile_reachable_population_total):
    adjusted = adjust_segment_percentages(indicator_pdf)
    total_home_count_sum = adjusted["total_home_count"].fillna(0).sum()
    if total_home_count_sum > 0:
        share = adjusted["total_home_count"].fillna(0) / total_home_count_sum
    else:
        share = pd.Series(np.zeros(len(adjusted)), index=adjusted.index)

    adjusted["estimated_adult_population"] = (adult_population_total * share).round(0).astype(int)
    adjusted["estimated_mobile_reachable_population"] = (mobile_reachable_population_total * share).round(0).astype(int)
    adjusted["adjusted_total_internet_non_mobile"] = (
        adjusted["adjusted_perc_internet_non_mobile"] / 100.0 * adjusted["estimated_adult_population"]
    ).round(0).astype(int)
    adjusted["adjusted_total_internet_ALL"] = (
        adjusted["adjusted_perc_internet"] / 100.0 * adjusted["estimated_adult_population"]
    ).round(0).astype(int)
    adjusted["adjusted_total_internet_mobile"] = (
        adjusted["adjusted_total_internet_ALL"] - adjusted["adjusted_total_internet_non_mobile"]
    ).clip(lower=0).round(0).astype(int)
    adjusted["adjusted_total_no_internet"] = (
        adjusted["estimated_adult_population"] - adjusted["adjusted_total_internet_ALL"]
    ).clip(lower=0).round(0).astype(int)

    for tech in TECHNOLOGY_LEVELS:
        adjusted[f"adjusted_total_internet_{tech}"] = (
            adjusted[f"adjusted_perc_internet_{tech}"] / 100.0 * adjusted["estimated_adult_population"]
        ).round(0).astype(int)

    return adjusted


def make_percent_colormap(caption):
    colormap = LinearColormap(MAP_GRADIENT, vmin=0, vmax=100)
    colormap.caption = caption
    return colormap


def plot_percentage_bars(
    data,
    category_col,
    value_col,
    title,
    subtitle="",
    hue=None,
    palette="Set2",
    figsize=(12, 10),
    category_order=None,
):
    plot_df = data.copy()
    plot_df[value_col] = plot_df[value_col].fillna(0).clip(0, 100)

    fig, ax = plt.subplots(figsize=figsize)

    legend_handles = None
    legend_labels = None
    legend_title = None
    if hue is None:
        if category_order is not None:
            plot_df[category_col] = pd.Categorical(plot_df[category_col], categories=category_order, ordered=True)
            plot_df = plot_df.sort_values(category_col, ascending=False)
        else:
            plot_df = plot_df.sort_values(value_col, ascending=True)
        colors = [make_percent_colormap("Share of adults (%)")(value) for value in plot_df[value_col]]
        bars = ax.barh(
            plot_df[category_col],
            plot_df[value_col],
            color=colors,
            edgecolor="white",
            linewidth=1.2,
        )
        labels = [f"{value:.1f}%" for value in plot_df[value_col]]
        ax.bar_label(bars, labels=labels, padding=4, fontsize=9)
    else:
        if category_order is None:
            order = (
                plot_df.groupby(category_col)[value_col]
                .max()
                .sort_values(ascending=False)
                .index
                .tolist()
            )
        else:
            order = list(category_order)
        sns.barplot(
            data=plot_df,
            y=category_col,
            x=value_col,
            hue=hue,
            order=order,
            orient="h",
            palette=palette,
            ax=ax,
        )
        for container in ax.containers:
            labels = [
                f"{bar.get_width():.1f}%" if bar.get_width() > 0 else ""
                for bar in container
            ]
            ax.bar_label(container, labels=labels, padding=3, fontsize=8)
        if ax.legend_ is not None:
            legend_handles, legend_labels = ax.get_legend_handles_labels()
            legend_title = hue.replace("_", " ").title()
            ax.legend_.remove()

    ax.set_xlim(0, 100)
    ax.xaxis.set_major_locator(MultipleLocator(10))
    ax.set_xlabel("Share of adults (%)")
    ax.set_ylabel(category_col.replace("_", " ").title())
    ax.grid(False)
    ax.margins(y=0.005)
    ax.tick_params(axis="y", length=0)
    sns.despine(left=True, bottom=True)
    wrapped_title = fill(title, 48)
    wrapped_subtitle = fill(subtitle, 110) if subtitle else ""
    title_lines = wrapped_title.count("\n") + 1
    subtitle_lines = wrapped_subtitle.count("\n") + 1 if subtitle else 0
    fig_height = fig.get_figheight()
    title_line_height = 0.24 / fig_height
    subtitle_line_height = 0.14 / fig_height
    pad_top = 0.08 / fig_height
    pad_between = 0.03 / fig_height if subtitle else 0
    pad_bottom = 0.02 / fig_height
    top = 1 - (
        pad_top
        + title_lines * title_line_height
        + pad_between
        + subtitle_lines * subtitle_line_height
        + pad_bottom
    )
    top = max(0.72, top)
    title_y = 1 - pad_top
    fig.text(0.5, title_y, wrapped_title, ha="center", va="top", fontsize=14, fontweight="bold")
    if subtitle:
        subtitle_y = title_y - title_lines * title_line_height - pad_between
        fig.text(
            0.5,
            subtitle_y,
            wrapped_subtitle,
            ha="center",
            va="top",
            fontsize=10,
            color="#4b5563",
        )
    fig.tight_layout()
    fig.subplots_adjust(top=top)
    if legend_handles:
        fig.legend(
            legend_handles,
            legend_labels,
            title=legend_title,
            frameon=False,
            loc="upper right",
            bbox_to_anchor=(0.98, 1 - pad_top),
            bbox_transform=fig.transFigure,
            borderaxespad=0,
        )
    return fig, ax


def map_center(geo_df):
    minx, miny, maxx, maxy = geo_df.total_bounds
    return [(miny + maxy) / 2, (minx + maxx) / 2]


def add_map_header(map_object, title, subtitle):
    header_html = f'''
    <div style="
        position: fixed;
        top: 18px;
        left: 60px;
        z-index: 9999;
        background: rgba(255, 255, 255, 0.96);
        padding: 12px 16px;
        border-radius: 10px;
        box-shadow: 0 4px 12px rgba(0, 0, 0, 0.18);
        min-width: 280px;
        font-family: Arial, sans-serif;
    ">
        <div style="font-size: 15px; font-weight: 700; color: #111827;">{title}</div>
        <div style="font-size: 12px; color: #4b5563; margin-top: 4px;">{subtitle}</div>
    </div>
    '''
    map_object.get_root().html.add_child(folium.Element(header_html))


def add_percentage_layer(
    map_object,
    geo_df,
    value_col,
    layer_name,
    tooltip_fields,
    tooltip_aliases,
    color_scale,
    show=False,
    overlay=True,
):
    layer = folium.FeatureGroup(name=layer_name, show=show, overlay=overlay)

    def style_function(feature):
        value = float(feature["properties"].get(value_col, 0) or 0)
        homes = float(feature["properties"].get("adult_population", 0) or 0)
        value = max(0.0, min(100.0, value))
        fill_color = NO_HOME_COLOR if homes <= 0 else color_scale(value)
        return {
            "fillColor": fill_color,
            "color": "#475569",
            "weight": 0.8,
            "fillOpacity": 0.88,
        }

    def highlight_function(feature):
        value = float(feature["properties"].get(value_col, 0) or 0)
        homes = float(feature["properties"].get("adult_population", 0) or 0)
        value = max(0.0, min(100.0, value))
        fill_color = NO_HOME_COLOR if homes <= 0 else color_scale(value)
        return {
            "fillColor": fill_color,
            "color": "#0f172a",
            "weight": 1.5,
            "fillOpacity": 0.95,
        }

    tooltip = folium.features.GeoJsonTooltip(
        fields=tooltip_fields,
        aliases=tooltip_aliases,
        localize=True,
        sticky=False,
        labels=True,
        style=(
            "background-color: white; color: #1f2937; font-family: Arial; "
            "font-size: 12px; padding: 10px; border: 1px solid #cbd5e1;"
        ),
    )

    folium.GeoJson(
        geo_df.to_json(),
        style_function=style_function,
        highlight_function=highlight_function,
        tooltip=tooltip,
    ).add_to(layer)

    layer.add_to(map_object)


def prepare_adjusted_geodata(adjusted_pdf, geojson_file):
    geo_df = load_admin_boundaries(geojson_file)
    adjusted_pdf = adjusted_pdf.copy()
    for column in ["municipality", "geomatch"]:
        adjusted_pdf[column] = adjusted_pdf[column].astype("string").str.strip()
    geo_df = geo_df.merge(adjusted_pdf, on=["municipality", "geomatch"], how="left")
    for column in ADJUSTED_PERCENT_COLUMNS:
        geo_df[column] = geo_df[column].fillna(0).clip(0, 100).round(2)
    for column in [
        "population_total",
        "adult_population",
        "mobile_reachable_population",
        "adjusted_total_internet_mobile",
        "adjusted_total_internet_non_mobile",
        "adjusted_total_internet_ALL",
    ]:
        if column in geo_df.columns:
            geo_df[column] = geo_df[column].fillna(0).round(0).astype(int)
    return prepare_web_map_geodata(geo_df)


## 4. Load the Saved Outputs from Notebook 04-Calculate indicator

The post-adjustment notebook starts from the final saved tables created earlier in the pipeline. This keeps the original subscription-based calculations intact and makes the adjustment stage auditable: the raw and adjusted indicators can be compared side by side without recomputing the MPD logic.

If `OPERATOR_CONFIGS` is empty, the notebook loads the tables from `INDICATOR_PATH` and applies the single-operator SIM duplication bias parameters. If `OPERATOR_CONFIGS` contains two or more operator folders, the notebook corrects each operator table using its own SIM factors and then combines those corrected percentages using normalized market-share weights before the population adjustment step.

The municipality table is required. Zone and CRM tables are optional and are only processed when the corresponding files exist.

In [ ]:
ind_adm = load_indicator_table("ind_adm.csv", required=True)
ind_zone = load_indicator_table("ind_zone.csv", required=False)
ind_adm_zone = load_indicator_table("ind_adm_zone.csv", required=False)
ind_age = ensure_age_group_order(load_indicator_table("ind_age.csv", required=False))
ind_gen = load_indicator_table("ind_gen.csv", required=False)
ind_gen_age = ensure_age_group_order(load_indicator_table("ind_gen_age.csv", required=False))

ZONE_FLAG = ind_zone is not None and ind_adm_zone is not None
CRM_FLAG = ind_age is not None and ind_gen is not None and ind_gen_age is not None

print(f"Operator mode: {OPERATOR_MODE}")
print(f"Operators configured: {[record['name'] for record in OPERATOR_CONFIG_RECORDS]}")
print(f"ZONE_FLAG={ZONE_FLAG}, CRM_FLAG={CRM_FLAG}")
print(f"Municipality rows loaded: {len(ind_adm):,}")
ind_adm.head()

## 5. Load the Administrative Boundaries and Resolve the WorldPop Source

The geographic aggregation in this notebook is anchored on `GEOJSON_FILE`. The same boundary file is used for both the population overlay and the final adjusted maps.

The WorldPop source is configured directly from `COUNTRY_CODE` in `script/conf.py`. The same configured country code is used to build the expected TIFF filename and the WorldPop download URL.


In [ ]:
admin_boundaries = load_admin_boundaries(GEOJSON_FILE)
country_code = COUNTRY_CODE.lower()
if not country_code:
    raise ValueError("COUNTRY_CODE must be set in script/conf.py before running notebook 05.")
pop_name, pop_url = build_worldpop_download(
    WORLDPOP_YEAR,
    country_code,
    WORLDPOP_DATABASE_URL,
    pop_res_1km=WORLDPOP_USE_1KM,
    pop_unadj=WORLDPOP_UNADJUSTED,
)
pop_path = DATA_DIR / pop_name
admin_bounds = admin_boundaries.total_bounds

print(f"WorldPop country code: {country_code.upper()}")
print(f"WorldPop raster path: {pop_path}")
print(f"WorldPop year: {WORLDPOP_YEAR}")
print(f"WorldPop URL: {pop_url}")
print(
    "Adjustment parameters -> "
    f"CHILDREN_PERC={CHILDREN_PERC}, "
    f"NO_PHONE_PERC={NO_PHONE_PERC}, "
    f"SIMS_PER_INTERNET_USER={SIMS_PER_INTERNET_USER}, "
    f"SIMS_PER_NON_INTERNET_USER={SIMS_PER_NON_INTERNET_USER}, "
    f"NON_MOBILE_PHONE_INTERNET_PERC={NON_MOBILE_PHONE_INTERNET_PERC}"
)
admin_boundaries.head()


## 6. Download or Reuse the WorldPop Raster

This notebook follows the same pattern as the synthetic baselayer workflow:

1. if the WorldPop TIFF already exists locally, reuse it,
2. otherwise download it from the resolved WorldPop URL,
3. open the raster and extract only the pixels that intersect the administrative extent.

Only positive-population pixels are retained, which keeps the spatial join efficient and avoids unnecessary processing outside the area of interest.


In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
if not pop_path.exists():
    print(f"Downloading WorldPop TIFF from {pop_url}")
    urllib.request.urlretrieve(pop_url, pop_path)
else:
    print(f"Using existing WorldPop TIFF: {pop_path}")

print("Extracting WorldPop population values inside the administrative bounds...")
df_pop = extract_worldpop_points(pop_path, admin_bounds)
print(f"Population points retained: {len(df_pop):,}")
print(f"Population sum inside bounds: {df_pop['population'].sum():,.2f}")
df_pop.head()

## 7. Aggregate WorldPop Population to the Administrative Areas

Each retained WorldPop pixel is converted to a point at the pixel center and spatially joined to the polygons from `GEOJSON_FILE`. The resulting table provides one population estimate per geographic area.

This is also the stage where the adult population and the mobile-reachable adult population are derived from `CHILDREN_PERC` and `NO_PHONE_PERC`.


In [ ]:
wp_gdf = gpd.GeoDataFrame(
    df_pop[["population"]].copy(),
    geometry=gpd.points_from_xy(df_pop.longitude, df_pop.latitude),
    crs="EPSG:4326",
)

population_matches = gpd.sjoin(
    wp_gdf,
    admin_boundaries[["municipality", "geomatch", "geometry"]],
    how="inner",
    predicate="within",
)

population_by_adm = (
    population_matches.groupby(["municipality", "geomatch"], as_index=False)["population"]
    .sum()
    .rename(columns={"population": "population_total"})
)

population_by_adm = (
    admin_boundaries[["municipality", "geomatch"]]
    .drop_duplicates()
    .merge(population_by_adm, on=["municipality", "geomatch"], how="left")
)
population_by_adm["population_total"] = population_by_adm["population_total"].fillna(0)
population_by_adm["adult_population"] = population_by_adm["population_total"] * (1 - CHILDREN_SHARE)
population_by_adm["mobile_reachable_population"] = population_by_adm["adult_population"] * (1 - NO_PHONE_SHARE)

for column in ["population_total", "adult_population", "mobile_reachable_population"]:
    population_by_adm[column] = population_by_adm[column].round(0).astype(int)

save_indicator_table(population_by_adm, "population_by_municipality.csv")
population_by_adm.sort_values("population_total", ascending=False).head()

## 8. Calculate the Adjusted Municipality Indicator Table

The municipality table from notebook `04` provides raw subscription-based internet-use counts and technology splits. Notebook `05` first corrects the mobile internet share for possible duplicate SIM ownership by converting internet and non-internet subscription counts into user-equivalent counts with `SIMS_PER_INTERNET_USER` and `SIMS_PER_NON_INTERNET_USER`. In multi-operator mode, this correction is applied separately to every configured operator before market-share weighting.

The resulting corrected mobile-user internet percentages are then adjusted for phone access and, if configured, augmented with a non-mobile-phone internet-use component before they are converted into adjusted population counts using the municipality adult-population table derived from WorldPop.

This keeps the logic explicit: the corrected mobile share is calculated first, the final adjusted adult-population share is calculated second, and the final adjusted number of users is calculated afterwards as `adult_population * final_internet_user_perc`.

In [ ]:
ind_adm_adjusted = adjust_area_indicator(ind_adm, population_by_adm)
save_indicator_table(ind_adm_adjusted, "ind_adm_adjusted.csv")
ind_adm_adjusted.head()

## 9. Bar Chart of Adjusted Internet Use by Municipality

The first chart mirrors notebook `04`, but now uses the post-adjusted municipality percentage. This makes it possible to compare the spatial ordering of internet use after the adult-population, phone-access, and SIM-ownership corrections are applied.


In [ ]:
plot_percentage_bars(
    ind_adm_adjusted,
    category_col="municipality",
    value_col="adjusted_perc_internet",
    title="Post-adjusted internet use by municipality",
    subtitle="Adjusted percentages are reported against the estimated adult population in each municipality.",
    figsize=(11, max(8, len(ind_adm_adjusted) * 0.7)),
)
plt.show()

## 10. Prepare the Municipality Geometry for Mapping

The adjusted municipality table is merged back onto the boundary geometry so the same corrected indicators can be reused across all maps.


In [ ]:
adm_boundaries_adjusted = prepare_adjusted_geodata(ind_adm_adjusted, GEOJSON_FILE)
adm_boundaries_adjusted.head()

### Map of Adjusted Internet Use by Municipality

This map reproduces the overall municipality internet-use view from notebook `04`, but the fill layer now uses the adjusted percentage of adults using the internet. Tooltips include both the WorldPop-based population counts and the adjusted internet-user counts.


In [ ]:
tooltip_fields = [
    "municipality",
    "population_total",
    "adult_population",
    "mobile_reachable_population",
    "adjusted_total_internet_ALL",
    "adjusted_perc_internet",
    "adjusted_perc_internet_mobile",
    "adjusted_perc_internet_non_mobile",
    "adjusted_perc_internet_2G",
    "adjusted_perc_internet_3G",
    "adjusted_perc_internet_4G",
    "adjusted_perc_internet_5G",
    "adjusted_perc_no_internet",
]
tooltip_aliases = [
    "Municipality: ",
    "Population (WorldPop): ",
    "Adult population: ",
    "Mobile-reachable adults: ",
    "Adjusted internet users: ",
    "Adjusted internet users (% adults): ",
    "Adjusted mobile internet users (% adults): ",
    "Adjusted non-mobile internet users (% adults): ",
    "Adjusted 2G users (% adults): ",
    "Adjusted 3G users (% adults): ",
    "Adjusted 4G users (% adults): ",
    "Adjusted 5G users (% adults): ",
    "Adjusted no internet (% adults): ",
]

overall_map = folium.Map(location=map_center(adm_boundaries_adjusted), zoom_start=8, tiles="CartoDB positron")
overall_scale = make_percent_colormap("Adjusted internet users (% of adults)")

add_percentage_layer(
    overall_map,
    adm_boundaries_adjusted,
    value_col="adjusted_perc_internet",
    layer_name="Adjusted overall internet users",
    tooltip_fields=tooltip_fields,
    tooltip_aliases=tooltip_aliases,
    color_scale=overall_scale,
    show=True,
)

add_map_header(
    overall_map,
    "Adjusted municipality internet usage",
    "Percent of adults estimated to use the internet after the structural population adjustments.",
)
overall_scale.add_to(overall_map)
folium.LayerControl(collapsed=False).add_to(overall_map)
overall_map.save(POST_ADJUSTMENT_PATH + "ITU_post_adjusted_internet_usage.html")
overall_map

### Map of Adjusted Internet Use by Technology

The technology map keeps the same layer-switching structure as notebook `04`, but each layer now represents the adjusted percentage of adults attributed to the selected highest observed mobile internet technology. Any configured non-mobile-phone internet users are included in the overall layer but are not assigned to the 2G, 3G, 4G, or 5G layers.


In [ ]:
technology_map = folium.Map(location=map_center(adm_boundaries_adjusted), zoom_start=8, tiles="CartoDB positron")
technology_scale = make_percent_colormap("Adjusted internet users by selected technology (% of adults)")

add_percentage_layer(
    technology_map,
    adm_boundaries_adjusted,
    value_col="adjusted_perc_internet",
    layer_name="All internet users",
    tooltip_fields=tooltip_fields,
    tooltip_aliases=tooltip_aliases,
    color_scale=technology_scale,
    show=True,
    overlay=False,
)

for tech in TECHNOLOGY_LEVELS:
    add_percentage_layer(
        technology_map,
        adm_boundaries_adjusted,
        value_col=f"adjusted_perc_internet_{tech}",
        layer_name=f"{tech} internet users",
        tooltip_fields=tooltip_fields,
        tooltip_aliases=tooltip_aliases,
        color_scale=technology_scale,
        show=False,
        overlay=False,
    )

add_map_header(
    technology_map,
    "Adjusted technology-filtered internet usage",
    "Use the layer control to switch between overall, 2G, 3G, 4G, and 5G adjusted internet-use rates.",
)
technology_scale.add_to(technology_map)
folium.LayerControl(collapsed=False).add_to(technology_map)
technology_map.save(POST_ADJUSTMENT_PATH + "ITU_post_adjusted_internet_usage_by_technology_filter.html")
technology_map

## 11. Adjusted Technology Composition Map

The composition map uses the adjusted municipality percentages to show how the adult population in each area is split between `No Internet`, `2G`, `3G`, `4G`, and `5G` after the post-adjustment step.


In [ ]:
composition_map = folium.Map(location=map_center(adm_boundaries_adjusted), zoom_start=8, tiles="CartoDB positron")

add_map_header(
    composition_map,
    "Adjusted technology composition by municipality",
    "Each marker shows the adjusted adult-population split of no internet, 2G, 3G, 4G, and 5G users.",
)

base_outline = folium.GeoJson(
    adm_boundaries_adjusted.to_json(),
    style_function=lambda feature: {
        "fillColor": NO_HOME_COLOR if float(feature["properties"].get("adult_population", 0) or 0) <= 0 else "#f8fafc",
        "color": "#94a3b8",
        "weight": 0.7,
        "fillOpacity": 0.35 if float(feature["properties"].get("adult_population", 0) or 0) <= 0 else 0.2,
    },
    tooltip=folium.features.GeoJsonTooltip(
        fields=tooltip_fields,
        aliases=tooltip_aliases,
        localize=True,
        sticky=False,
        labels=True,
        style=(
            "background-color: white; color: #1f2937; font-family: Arial; "
            "font-size: 12px; padding: 10px; border: 1px solid #cbd5e1;"
        ),
    ),
)
base_outline.add_to(composition_map)

composition_df = ind_adm_adjusted[
    [
        "geomatch",
        "municipality",
        "adjusted_perc_no_internet",
        "adjusted_perc_internet_non_mobile",
        "adjusted_perc_internet_2G",
        "adjusted_perc_internet_3G",
        "adjusted_perc_internet_4G",
        "adjusted_perc_internet_5G",
    ]
].copy()

composition_long = composition_df.melt(
    id_vars=["geomatch", "municipality"],
    value_vars=[
        "adjusted_perc_no_internet",
        "adjusted_perc_internet_non_mobile",
        "adjusted_perc_internet_2G",
        "adjusted_perc_internet_3G",
        "adjusted_perc_internet_4G",
        "adjusted_perc_internet_5G",
    ],
    var_name="category",
    value_name="value",
)

composition_long["category"] = composition_long["category"].replace(
    {
        "adjusted_perc_no_internet": "No Internet",
        "adjusted_perc_internet_non_mobile": "Non-mobile",
        "adjusted_perc_internet_2G": "2G",
        "adjusted_perc_internet_3G": "3G",
        "adjusted_perc_internet_4G": "4G",
        "adjusted_perc_internet_5G": "5G",
    }
)
composition_long["value"] = composition_long["value"].fillna(0).clip(0, 100)

for geomatch, subset in composition_long.groupby("geomatch"):
    geometry = adm_boundaries_adjusted.loc[adm_boundaries_adjusted["geomatch"] == geomatch, "geometry"]
    if geometry.empty:
        continue

    values = subset["value"].tolist()
    if sum(values) <= 0:
        continue

    centroid = geometry.iloc[0].centroid
    colors = [TECH_COLORS[label] for label in subset["category"]]

    fig, ax = plt.subplots(figsize=(1.4, 1.4))
    ax.pie(
        values,
        startangle=90,
        colors=colors,
        wedgeprops={"width": 0.42, "edgecolor": "white"},
    )
    ax.set(aspect="equal")
    ax.axis("off")

    buffer = io.BytesIO()
    plt.savefig(buffer, format="png", bbox_inches="tight", transparent=True)
    plt.close(fig)

    image_b64 = base64.b64encode(buffer.getvalue()).decode("utf-8")
    html = f'<img src="data:image/png;base64,{image_b64}" width="62" height="62">'

    folium.Marker(
        location=[centroid.y, centroid.x],
        icon=folium.DivIcon(html=html),
    ).add_to(composition_map)

legend_html = '''
<div style="
    position: fixed;
    bottom: 22px;
    right: 22px;
    z-index: 9999;
    background: rgba(255, 255, 255, 0.96);
    padding: 10px 12px;
    border-radius: 10px;
    box-shadow: 0 4px 12px rgba(0, 0, 0, 0.18);
    font-family: Arial, sans-serif;
    font-size: 12px;
    min-width: 155px;
">
    <div style="font-weight: 700; margin-bottom: 8px;">Adjusted technology mix</div>
    <div><span style="display:inline-block;width:12px;height:12px;background:#b2182b;margin-right:6px;"></span>No Internet</div>
    <div><span style="display:inline-block;width:12px;height:12px;background:#4e79a7;margin-right:6px;"></span>Non-mobile</div>
    <div><span style="display:inline-block;width:12px;height:12px;background:#f28e2b;margin-right:6px;"></span>2G</div>
    <div><span style="display:inline-block;width:12px;height:12px;background:#edc948;margin-right:6px;"></span>3G</div>
    <div><span style="display:inline-block;width:12px;height:12px;background:#59a14f;margin-right:6px;"></span>4G</div>
    <div><span style="display:inline-block;width:12px;height:12px;background:#1a9850;margin-right:6px;"></span>5G</div>
</div>
'''
composition_map.get_root().html.add_child(folium.Element(legend_html))
composition_map.save(POST_ADJUSTMENT_PATH + "ITU_post_adjusted_internet_usage_by_technology.html")
composition_map

## 12. Optional Zone Outputs

When the zone tables from notebook `04` are available, the municipality adult population is allocated to `URBAN` and `RURAL` segments using the observed anchored subscriber share within each municipality. The adjusted percentage is then calculated first for each zone segment and converted into adjusted users afterwards. This keeps the municipality totals aligned with WorldPop while still reusing the existing zone structure from the indicator workflow.


In [ ]:
if ZONE_FLAG:
    municipality_population = ind_adm_adjusted[
        ["municipality", "geomatch", "adult_population", "mobile_reachable_population", "total_home_count"]
    ].rename(columns={"total_home_count": "municipality_total_home_count"})

    ind_adm_zone_adjusted = ind_adm_zone.merge(
        municipality_population,
        on="municipality",
        how="left",
    )
    ind_adm_zone_adjusted["zone_home_share"] = np.where(
        ind_adm_zone_adjusted["municipality_total_home_count"] > 0,
        ind_adm_zone_adjusted["total_home_count"] / ind_adm_zone_adjusted["municipality_total_home_count"],
        0,
    )
    ind_adm_zone_adjusted["estimated_adult_population"] = (
        ind_adm_zone_adjusted["adult_population"] * ind_adm_zone_adjusted["zone_home_share"]
    ).round(0).astype(int)
    ind_adm_zone_adjusted["estimated_mobile_reachable_population"] = (
        ind_adm_zone_adjusted["mobile_reachable_population"] * ind_adm_zone_adjusted["zone_home_share"]
    ).round(0).astype(int)

    ind_adm_zone_adjusted = adjust_segment_percentages(ind_adm_zone_adjusted)
    ind_adm_zone_adjusted["adjusted_total_internet_non_mobile"] = (
        ind_adm_zone_adjusted["adjusted_perc_internet_non_mobile"] / 100.0
        * ind_adm_zone_adjusted["estimated_adult_population"]
    ).round(0).astype(int)
    ind_adm_zone_adjusted["adjusted_total_internet_ALL"] = (
        ind_adm_zone_adjusted["adjusted_perc_internet"] / 100.0 * ind_adm_zone_adjusted["estimated_adult_population"]
    ).round(0).astype(int)
    ind_adm_zone_adjusted["adjusted_total_internet_mobile"] = (
        ind_adm_zone_adjusted["adjusted_total_internet_ALL"] - ind_adm_zone_adjusted["adjusted_total_internet_non_mobile"]
    ).clip(lower=0).round(0).astype(int)
    ind_adm_zone_adjusted["adjusted_total_no_internet"] = (
        ind_adm_zone_adjusted["estimated_adult_population"] - ind_adm_zone_adjusted["adjusted_total_internet_ALL"]
    ).clip(lower=0).round(0).astype(int)

    for tech in TECHNOLOGY_LEVELS:
        ind_adm_zone_adjusted[f"adjusted_total_internet_{tech}"] = (
            ind_adm_zone_adjusted[f"adjusted_perc_internet_{tech}"] / 100.0
            * ind_adm_zone_adjusted["estimated_adult_population"]
        ).round(0).astype(int)

    save_indicator_table(ind_adm_zone_adjusted, "ind_adm_zone_adjusted.csv")

    zone_group_cols = ["zone_classification"]
    zone_sum_cols = [
        "estimated_adult_population",
        "estimated_mobile_reachable_population",
        "adjusted_total_internet_mobile",
        "adjusted_total_internet_non_mobile",
        "adjusted_total_internet_ALL",
        "adjusted_total_no_internet",
    ] + [f"adjusted_total_internet_{tech}" for tech in TECHNOLOGY_LEVELS]

    ind_zone_adjusted = (
        ind_adm_zone_adjusted.groupby(zone_group_cols, as_index=False)[zone_sum_cols]
        .sum()
    )
    ind_zone_adjusted["adjusted_perc_internet"] = np.where(
        ind_zone_adjusted["estimated_adult_population"] > 0,
        ind_zone_adjusted["adjusted_total_internet_ALL"] / ind_zone_adjusted["estimated_adult_population"] * 100,
        0,
    )
    ind_zone_adjusted["adjusted_perc_internet_mobile"] = np.where(
        ind_zone_adjusted["estimated_adult_population"] > 0,
        ind_zone_adjusted["adjusted_total_internet_mobile"] / ind_zone_adjusted["estimated_adult_population"] * 100,
        0,
    )
    ind_zone_adjusted["adjusted_perc_internet_non_mobile"] = np.where(
        ind_zone_adjusted["estimated_adult_population"] > 0,
        ind_zone_adjusted["adjusted_total_internet_non_mobile"] / ind_zone_adjusted["estimated_adult_population"] * 100,
        0,
    )
    ind_zone_adjusted["adjusted_perc_no_internet"] = np.where(
        ind_zone_adjusted["estimated_adult_population"] > 0,
        ind_zone_adjusted["adjusted_total_no_internet"] / ind_zone_adjusted["estimated_adult_population"] * 100,
        0,
    )
    for tech in TECHNOLOGY_LEVELS:
        ind_zone_adjusted[f"adjusted_perc_internet_{tech}"] = np.where(
            ind_zone_adjusted["estimated_adult_population"] > 0,
            ind_zone_adjusted[f"adjusted_total_internet_{tech}"] / ind_zone_adjusted["estimated_adult_population"] * 100,
            0,
        )

    for column in ADJUSTED_PERCENT_COLUMNS:
        ind_zone_adjusted[column] = ind_zone_adjusted[column].fillna(0).clip(0, 100).round(2)

    save_indicator_table(ind_zone_adjusted, "ind_zone_adjusted.csv")

    plot_percentage_bars(
        ind_adm_zone_adjusted,
        category_col="municipality",
        value_col="adjusted_perc_internet",
        hue="zone_classification",
        title="Post-adjusted internet use by municipality and zone classification",
        subtitle="Zone populations are allocated from municipality adult populations using the observed anchored subscriber zone split.",
        figsize=(12, max(5.5, len(ind_adm_zone_adjusted["municipality"].unique()) * 0.9)),
    )
    plt.show()

    ind_zone_adjusted
else:
    print("Zone outputs skipped because the saved zone tables from notebook 04 are unavailable.")

## 13. Optional CRM Outputs by Age and Gender

When the CRM-based outputs from notebook `04` are available, the adjusted percentages are calculated on the same structural basis as the municipality outputs.

Because this notebook does not load an external age-by-sex population table, the adjusted counts for age and gender are allocated from the total adult population using the observed subscriber composition in the CRM tables. The adjusted percentage is still calculated first from the raw MPD percentage, and the calibrated counts are derived afterwards from the allocated adult population. Those counts should therefore be read as calibrated proxies, while the adjusted percentages remain the primary analytical outputs.


In [ ]:
if CRM_FLAG:
    adult_population_total = int(population_by_adm["adult_population"].sum())
    mobile_reachable_population_total = int(population_by_adm["mobile_reachable_population"].sum())

    ind_age_adjusted = allocate_population_from_subscriber_share(
        ind_age,
        adult_population_total=adult_population_total,
        mobile_reachable_population_total=mobile_reachable_population_total,
    )
    save_indicator_table(ind_age_adjusted, "ind_age_adjusted.csv")

    ind_gen_adjusted = allocate_population_from_subscriber_share(
        ind_gen,
        adult_population_total=adult_population_total,
        mobile_reachable_population_total=mobile_reachable_population_total,
    )
    save_indicator_table(ind_gen_adjusted, "ind_gen_adjusted.csv")

    ind_gen_age_adjusted = allocate_population_from_subscriber_share(
        ind_gen_age,
        adult_population_total=adult_population_total,
        mobile_reachable_population_total=mobile_reachable_population_total,
    )
    save_indicator_table(ind_gen_age_adjusted, "ind_gen_age_adjusted.csv")

    # plot_percentage_bars(
    #     ind_age_adjusted,
    #     category_col="age_group",
    #     value_col="adjusted_perc_internet",
    #     title="Post-adjusted internet use by age group",
    #     subtitle="Adjusted percentages are shown against the adult population framework used in this notebook.",
    #     figsize=(10, max(5.5, len(ind_age_adjusted) * 0.8)),
    #     category_order=(
    #         ind_age_adjusted[["age_group_order", "age_group"]]
    #         .drop_duplicates()
    #         .sort_values("age_group_order")["age_group"]
    #         .tolist()
    #     ),
    # )
    # plt.show()

    # plot_percentage_bars(
    #     ind_gen_adjusted,
    #     category_col="gender",
    #     value_col="adjusted_perc_internet",
    #     title="Post-adjusted internet use by gender",
    #     subtitle="The gender view uses the same structural adjustment factor and an adult-population proxy allocation for counts.",
    #     figsize=(9, 4.5),
    # )
    # plt.show()

    plot_percentage_bars(
        ind_gen_age_adjusted,
        category_col="age_group",
        value_col="adjusted_perc_internet",
        hue="gender",
        title="Post-adjusted internet use by age group and gender",
        subtitle="Percent of adults estimated to use the internet within each CRM segment after the post-adjustment step.",
        figsize=(11, max(6, len(ind_gen_age_adjusted["age_group"].unique()) * 0.8)),
        category_order=(
            ind_gen_age_adjusted[["age_group_order", "age_group"]]
            .drop_duplicates()
            .sort_values("age_group_order")["age_group"]
            .tolist()
        ),
    )
    plt.show()

    ind_gen_age_adjusted.head()
else:
    print("CRM outputs skipped because the saved CRM tables from notebook 04 are unavailable.")

## Implementation Notes

The post-adjustment workflow is intentionally separate from notebook `04`.

- The raw MPD-based indicator tables remain unchanged and are reused as auditable inputs.
- The WorldPop raster is only used to estimate population totals by the administrative areas in `GEOJSON_FILE`.
- `CHILDREN_PERC`, `NO_PHONE_PERC`, `SIMS_PER_INTERNET_USER`, `SIMS_PER_NON_INTERNET_USER`, and `NON_MOBILE_PHONE_INTERNET_PERC` are applied transparently and can be changed centrally in `script/conf.py`.
- `SIMS_PER_INTERNET_USER` and `SIMS_PER_NON_INTERNET_USER` are applied to count columns from notebook `04`, not directly to an already-normalized percentage. If both values are equal, the differential SIM correction cancels out.
- `OPERATOR_CONFIGS` can be left empty for a single operator or populated with multiple notebook `04` output folders to produce a market-share-weighted multi-operator adjustment. Each config may use `indicator_path` or `indicator_dir` for the notebook `04` output folder.
- Municipality outputs first correct the mobile subscription sample for differential SIM ownership and only then derive final adjusted user counts from the adult population.
- Zone outputs inherit municipality population through the observed zone split of anchored subscribers.
- Age and gender outputs reuse the CRM distributions from notebook `04` and allocate calibrated counts from the adult population total because no external demographic population table is introduced here.

This structure keeps the post-adjustment stage explicit, reproducible, and easy to compare against the original subscription-based outputs.